<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/dots.tts-soar_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## dots.tts-soar — 48 kHz Continuous-VAE TTS with SOTA Multilingual SIM

2B-parameter fully-continuous autoregressive TTS from Red Note (\u5c0f\u7ea2\u4e66) Lab. Uses a frozen AudioVAE + BigVGAN-style decoder (no discrete tokens anywhere in the pipeline) for **48 kHz studio-quality output**. Achieves the **best average speaker similarity (83.9) on the 24-language MiniMax multilingual benchmark** and best avg SIM (79.2) on Seed-TTS-Eval among open-source systems. Apache 2.0.

### Quick Start
1. Connect to a GPU runtime (T4 16 GB works, L4/A100 recommended)
2. Run Steps 1\u20134 in order \u2014 no token, no sign-up needed
3. Open the Gradio UI link from Step 4 and start generating

| GPU | VRAM | Status |
|-----|------|--------|
|-----|------|--------|
| A100 | 40 GB | Fastest |
|-----|------|--------|
| L4 | 24 GB | Recommended |
|-----|------|--------|
| T4 | 16 GB | Works in bf16 (\u22484 GB weights) |

**First run downloads ~9.5 GB** (model + vocoder + speaker encoder).

### How to Use
dots.tts-soar is **voice-cloning only** \u2014 there are no fixed voices. For each generation you either:
- **Continuation cloning (recommended):** Upload a 5\u201310s reference clip **and** provide its transcript. Highest fidelity, captures prosody and style.
- **X-vector-only cloning:** Upload just the reference audio, no transcript. Faster, but lower fidelity.
- **Random-voice sampling:** Leave the reference empty. Only meaningful on fine-tuned single-speaker checkpoints \u2014 with the public zero-shot model, results are nondescript.

### Generation Tips
- **`--num-steps` 10** is the sweet spot for the SCA checkpoint (default in this notebook). Higher = slightly better quality, much slower.
- **`--guidance-scale` 1.2** is recommended for SCA \u2014 the model already tightens text/timbre alignment, so small CFG suffices.
- **Use `--language`** to force a language tag: `EN`, `ZH`, `Cantonese`, or `auto_detect` to let the model infer.
- **`--normalize-text`** turns on text normalization (numbers, currency, dates). Off by default to give you control.

**License:** Apache 2.0. The model card prohibits using the model for impersonation, fraud, or disinformation \u2014 only use it for research and authorized deployment.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Clones the dots.tts repo and installs runtime deps (skipping torch downgrade).

import os, sys, subprocess
import torch

if not torch.cuda.is_available():
    raise SystemExit('No GPU detected. Connect a GPU runtime: Runtime -> Change runtime type -> T4 / L4 / A100')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'[GPU] {gpu_name} — {vram_gb:.1f} GB')

REPO_DIR = '/content/dots.tts'
REPO_URL = 'https://github.com/rednote-hilab/dots.tts.git'

if not os.path.isdir(REPO_DIR):
    print(f'[git] Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    print(f'[git] {REPO_DIR} already present, skipping clone.')

# constraints/recommended.txt pins torch==2.8.0 — incompatible with Colab torch 2.11
print('[pip] Installing dots.tts (--no-deps, keeping Colab torch 2.11) ...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', REPO_DIR],
    check=True,
)

# Runtime deps from pyproject.toml (skip torch/torchaudio/transformers — Colab ships them)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'huggingface-hub', 'loguru', 'langcodes[data]', 'einops',
     'librosa>=0.11.0', 'soundfile>=0.13.1',
     'pydantic>=2.12.5,<3', 'PyYAML>=6.0.3', 'safetensors>=0.8.0rc0',
     'torchdiffeq', 'tqdm', 'lingua-language-detector', 'WeTextProcessing',
     'accelerate>=1.12.0', 'tensorboard>=2.20.0',
     'gradio==5.33.0', 'hf_transfer'],
    check=True,
)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
src_root = os.path.join(REPO_DIR, 'src')
if src_root not in sys.path:
    sys.path.insert(0, src_root)
print(f'[sys.path] Added {REPO_DIR} and {src_root}')

os.makedirs('/content/audio_out', exist_ok=True)
os.makedirs('/content/ref_audio', exist_ok=True)
print('[Setup] /content/audio_out and /content/ref_audio ready.')


In [ ]:
# @title STEP 2 — Pre-cache Weights
# @markdown Downloads the dots.tts-soar weights (\u22489.5 GB total: model + vocoder + speaker encoder). The dots.tts repo does not ship bundled voice samples — you'll need to upload your own for voice cloning.

# @markdown Reuses the Drive-cached weights from `TTS_Model_Loader.ipynb` if present,
# @markdown otherwise downloads to the default ~/.cache/huggingface (in-session only).
import pathlib
try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_TTS_Cache')
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault('HF_HOME', str(CACHE_ROOT / 'huggingface'))
    print(f'[Cache] using Drive at {CACHE_ROOT}')
except Exception:
    print('[Cache] Drive not available, using default ~/.cache/huggingface')


import os
from huggingface_hub import snapshot_download

REPO = 'rednote-hilab/dots.tts-soar'
print(f'[Download] {REPO} (\u22489.5 GB) ...')
snapshot_download(REPO)
print('[Download] All weights cached.')

In [ ]:
# @title STEP 3 — Core Functions (lazy model loading)
# @markdown Wraps the `DotsTtsRuntime` class. Model loads on first use, then stays in memory.

import os, time, tempfile
import numpy as np
import torch
import soundfile as sf

REPO_ID = 'rednote-hilab/dots.tts-soar'
_RUNTIME = None
_SAMPLE_RATE = None

def get_runtime():
    global _RUNTIME, _SAMPLE_RATE
    if _RUNTIME is not None:
        return _RUNTIME
    from dots_tts.runtime import DotsTtsRuntime
    print(f'[Load] {REPO_ID} (bfloat16) ...')
    t0 = time.time()
    _RUNTIME = DotsTtsRuntime.from_pretrained(REPO_ID, precision='bfloat16')
    # The runtime doesn't expose sample_rate directly; do a quick warmup gen to learn it
    _ = _RUNTIME.generate(text='warmup.', prompt_audio_path=None, prompt_text=None,
                          num_steps=2, guidance_scale=1.0, seed=0)
    _SAMPLE_RATE = int(_['sample_rate'])
    print(f'[Load] Ready in {time.time()-t0:.1f}s — sample_rate={_SAMPLE_RATE} Hz')
    return _RUNTIME

def _materialize_ref_audio(ref_audio_path):
    if isinstance(ref_audio_path, tuple) and len(ref_audio_path) == 2:
        sr, audio = ref_audio_path
        tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        sf.write(tmp.name, audio, sr, format='WAV')
        tmp.flush()
        return tmp.name
    return ref_audio_path

def synth(text: str,
          ref_audio_path=None,
          ref_text: str = '',
          num_steps: int = 10,
          guidance_scale: float = 1.2,
          normalize_text: bool = False,
          language: str = 'none',
          seed: int = 42,
          out_name: str = 'dots.wav') -> tuple[str, int]:
    """Run a single dots.tts generation. Returns (path, sample_rate)."""
    r = get_runtime()
    text = (text or '').strip()
    if not text:
        raise RuntimeError('Text is empty.')

    kwargs = dict(
        text=text,
        num_steps=int(num_steps),
        guidance_scale=float(guidance_scale),
        normalize_text=bool(normalize_text),
        seed=int(seed),
    )
    if ref_audio_path:
        kwargs['prompt_audio_path'] = _materialize_ref_audio(ref_audio_path)
        # Continuation cloning needs the transcript; x-vector-only is used when ref_text is empty
        if ref_text and ref_text.strip():
            kwargs['prompt_text'] = ref_text.strip()
    if language and language != 'none':
        kwargs['language'] = language

    result = r.generate(**kwargs)
    out_path = os.path.join('/content/audio_out', out_name)
    audio_np = result['audio'].float().cpu().squeeze().numpy()
    sr = int(result['sample_rate'])
    sf.write(out_path, audio_np, sr)
    return out_path, sr

print('[Core] Ready. Use Step 4 for the UI, Step 6 for quick test, Step 7 for batch.')

In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app. Upload a reference clip (and its transcript for best results), pick a language, type text, hit Generate. Click the `.gradio.live` link to open.

import sys
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass
import gradio as gr

LANGUAGES = [
    ('Auto-detect', 'auto_detect'),
    ('English', 'EN'),
    ('Chinese (Mandarin)', 'ZH'),
    ('Cantonese', 'Cantonese'),
    ('No language tag', 'none'),
]
LANG_DISPLAY = [name for name, _ in LANGUAGES]
LANG_CODES = dict(LANGUAGES)

def _err(msg):
    return None, f'\u274c {msg}'

def ui_synth(text, ref_audio, ref_text, x_vector_only, language,
             num_steps, guidance_scale, normalize_text, seed, progress=gr.Progress()):
    if not text or not text.strip():
        return _err('Type some text first.')
    if ref_audio is None:
        return _err('Upload a reference voice clip — dots.tts is voice-cloning only.')
    if x_vector_only:
        effective_ref_text = ''
    else:
        effective_ref_text = (ref_text or '').strip()
        if not effective_ref_text:
            return _err('Provide the reference transcript, or enable x-vector-only mode.')
    try:
        progress(0.1, desc='Loading runtime (first run is slow)...')
        get_runtime()
        progress(0.4, desc='Synthesizing...')
        path, sr = synth(
            text.strip(),
            ref_audio_path=ref_audio,
            ref_text=effective_ref_text,
            num_steps=int(num_steps),
            guidance_scale=float(guidance_scale),
            normalize_text=bool(normalize_text),
            language=LANG_CODES[language],
            seed=int(seed),
            out_name='ui.wav',
        )
        progress(1.0, desc='Done.')
        return path, f'\u2705 Generated @ {sr} Hz'
    except Exception as e:
        return None, f'\u274c {e}'

DEFAULT_TEXT = 'I am solving the equation: x equals negative b, plus or minus the square root of b squared minus four a c, all divided by two a.'
DEFAULT_REF_TEXT = 'The quick brown fox jumps over the lazy dog. Pack my box with five dozen liquor jugs.'

with gr.Blocks(title='dots.tts-soar', theme=gr.themes.Soft()) as demo:
    gr.Markdown(f'# \ud83d\udccd dots.tts-soar\n**Model:** `rednote-hilab/dots.tts-soar` (2B, bf16) \u00b7 **GPU:** {gpu_name} ({vram_gb:.0f} GB)\n\nFully-continuous autoregressive TTS. 48 kHz output, voice cloning from 5\u201310s reference clip. Highest avg SIM (83.9) on the 24-language MiniMax benchmark.')
    with gr.Row():
        with gr.Column():
            text = gr.Textbox(
                label='Text to synthesize',
                value=DEFAULT_TEXT,
                lines=4,
                placeholder='Type what you want the voice to say...',
            )
            with gr.Accordion('Voice cloning (required)', open=True):
                ref_audio = gr.Audio(
                    label='Reference voice (5\u201310 seconds, single speaker, no background music)',
                    type='filepath', info="5-10s single-speaker, no background music. 48 kHz output, no discrete tokens.",
                    )
                ref_text = gr.Textbox(
                    label='Reference transcript (required for continuation cloning — high fidelity)',
                    value=DEFAULT_REF_TEXT,
                    lines=2,
                )
                x_vector_only = gr.Checkbox(
                    label='x-vector only (skip transcript — faster, lower fidelity)',
                    value=False,
                )
            with gr.Row():
                language = gr.Dropdown(LANG_DISPLAY, value='Auto-detect', label='Language tag')
            with gr.Accordion('Advanced', open=False):
                num_steps = gr.Slider(1, 32, value=10, step=1, label='Num steps (10 is sweet spot for SCA)', info="Flow-matching sampling steps. 10 is the sweet spot for SCA-quality output.")
                guidance_scale = gr.Slider(1.0, 3.0, value=1.2, step=0.1, label='Guidance scale (1.2 recommended)', info="CFG for the AR flow head. 1.0 = no guidance, 1.2 = recommended.")
                normalize_text = gr.Checkbox(label='Normalize text (numbers, currency, dates)', value=False)
                seed = gr.Number(value=42, label='Seed', precision=0)
            btn = gr.Button('Generate speech', variant='primary', size='lg')
        with gr.Column():
            output_audio = gr.Audio(label='Generated speech (48 kHz)', type='filepath')
            status = gr.Markdown('')
    gr.Examples(
        examples=[
            ['It was the best of times, it was the worst of times.', None, DEFAULT_REF_TEXT, False, 'Auto-detect'],
            ['The quick brown fox jumps over the lazy dog.', None, DEFAULT_REF_TEXT, False, 'Auto-detect'],
            ['To be, or not to be: that is the question.', None, DEFAULT_REF_TEXT, False, 'Auto-detect'],
            ['She sells seashells by the seashore.', None, DEFAULT_REF_TEXT, True,  'Auto-detect'],
        ],
        inputs=[text, ref_audio, ref_text, x_vector_only, language],
        label='Preloaded example texts (you still need to upload a reference clip to generate)',
    )
    gr.Markdown('**Tips:** For highest quality, use **continuation cloning** (upload reference + provide its transcript). For convenience, enable **x-vector-only** mode to skip the transcript. **48 kHz output** is studio-grade — perfect for downstream production. See the [dots.tts paper](https://huggingface.co/rednote-hilab/dots.tts-soar) for the full architecture and benchmark tables.')
    btn.click(
        ui_synth,
        inputs=[text, ref_audio, ref_text, x_vector_only, language, num_steps, guidance_scale, normalize_text, seed],
        outputs=[output_audio, status],
    )

from IPython.display import clear_output as _clear
_clear()
demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name="0.0.0.0", server_port=7860,
)
demo.load(lambda: "🎙️ dots.tts-soar ready. Upload a ref clip + transcript for voice cloning.", outputs=[status])


In [ ]:
# @title Step 5 — Keep Alive
# @markdown Prevents Colab from disconnecting during long generation sessions. Run anytime after launching the UI.

import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')

In [ ]:
# @title Step 6 — Quick Test (one-off inference, no UI)
# @markdown Run a single TTS call from the notebook. **You must provide a reference audio path** (and ideally its transcript) for the model to clone.

TEXT = "I am solving the equation x equals negative b, plus or minus the square root of b squared minus four a c, all divided by two a." # @param {type:"string"}
REF_AUDIO = "" # @param {type:"string"}
REF_TEXT = "" # @param {type:"string"}
X_VECTOR_ONLY = False # @param {type:"boolean"}
LANGUAGE = "auto_detect" # @param ["auto_detect", "EN", "ZH", "Cantonese", "none"]
NUM_STEPS = 10 # @param {type:"integer"}
GUIDANCE_SCALE = 1.2 # @param {type:"number"}
NORMALIZE_TEXT = False # @param {type:"boolean"}
SEED = 42 # @param {type:"integer"}

if not REF_AUDIO:
    raise SystemExit('REF_AUDIO is required. Upload a voice sample and set REF_AUDIO to its path.')
effective_ref_text = '' if X_VECTOR_ONLY else (REF_TEXT or '').strip()
if not X_VECTOR_ONLY and not effective_ref_text:
    raise SystemExit('REF_TEXT is required for continuation cloning. Enable X_VECTOR_ONLY to skip it.')

path, sr = synth(
    TEXT,
    ref_audio_path=REF_AUDIO,
    ref_text=effective_ref_text,
    num_steps=NUM_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    normalize_text=NORMALIZE_TEXT,
    language=LANGUAGE,
    seed=SEED,
)
print(f'[Done] {path} ({sr} Hz)')
from IPython.display import Audio, display
display(Audio(path))

In [ ]:
# @title Step 7 — Batch Synthesis
# @markdown Generate audio for a list of texts. Each line in `BATCH` becomes one output file. **All lines use the same reference voice.**

BATCH_REF_AUDIO = '' # @param {type:"string"}
BATCH_REF_TEXT = '' # @param {type:"string"}
BATCH_X_VECTOR_ONLY = False # @param {type:"boolean"}
BATCH_LANGUAGE = 'auto_detect' # @param ["auto_detect", "EN", "ZH", "Cantonese", "none"]
BATCH_NUM_STEPS = 10 # @param {type:"integer"}
BATCH_GUIDANCE_SCALE = 1.2 # @param {type:"number"}
BATCH_NORMALIZE_TEXT = False # @param {type:"boolean"}
BATCH_SEED = 42 # @param {type:"integer"}

BATCH = """\
The quick brown fox jumps over the lazy dog.
She sells seashells by the seashore.
To be, or not to be: that is the question.
It was the best of times, it was the worst of times.
All happy families are alike; each unhappy family is unhappy in its own way.
"""

if not BATCH_REF_AUDIO:
    raise SystemExit('BATCH_REF_AUDIO is required.')
effective_ref_text = '' if BATCH_X_VECTOR_ONLY else (BATCH_REF_TEXT or '').strip()

lines = [l.strip() for l in BATCH.strip().splitlines() if l.strip()]
out_dir = '/content/audio_out/batch'
os.makedirs(out_dir, exist_ok=True)

for i, line in enumerate(lines):
    try:
        print(f'[{i+1}/{len(lines)}] {line[:60]}{"..." if len(line) > 60 else ""}')
        path, sr = synth(
            line,
            ref_audio_path=BATCH_REF_AUDIO,
            ref_text=effective_ref_text,
            num_steps=BATCH_NUM_STEPS,
            guidance_scale=BATCH_GUIDANCE_SCALE,
            normalize_text=BATCH_NORMALIZE_TEXT,
            language=BATCH_LANGUAGE,
            seed=BATCH_SEED,
            out_name=f'{i:03d}.wav',
        )
        os.rename(path, os.path.join(out_dir, f'{i:03d}.wav'))

    except Exception as e:
        print(f"  -> EXCEPTION: {e}")
        continue
print(f'\n[Done] {len(lines)} files written to {out_dir}')